In [1]:
import spacy

In [2]:
nlp = spacy.load('en_core_web_md')

In [ ]:
# ,"D000003886256*REALMANAGE LLC*6400 INTERNATIONAL PARKWAY STE 1000*PLANO TX 75093* 0 PLANO TX 75093 US /"
# ,"PAYCOM PAYROLL LLC 7501 W MEMORIAL RD OKLAHOMA CITY OK 73142 US /"
# ,"7101 CARDINAL MANAGEMENT GROUP INC 4330 PRINCE WILLIAM PKWY STE 201 WOO DBRIDGE VA 22192 /"
# ,"3596814 14785 PSS INDUSTRIAL GROUP CORP 10507 ELLA BLVD S TE 100 HOUSTON TX 77038 I/US"
# ,"A72BRWA00000000010 00012 SHERIF ABDELFATTAH O/"

for delim in ['*','/','-']:
    text=text.replace(delim,f' {delim} ')

print(text)
doc = nlp(text)


A72BRWA00000000010 00012 SHERIF ABDELFATTAH O / 


In [68]:
import re

LEGAL_SUFFIXES = {
    "LLC", "L.L.C",
    "INC", "INC.",
    "LTD", "LTD.",
    "LLP",
    "CORP", "CORPORATION",
    "CO", "CO.",
    "COMPANY",
    "HOLDINGS", "GROUP"
}

O_MARKER_RE = re.compile(r"O/\d*/|O/")

STOP_WORDS = {
    "NOTPROVIDED", "NA", "N/A", "UNKNOWN", "UNAVAILABLE"
}

def getEntity(doc):
    
    # 1. ORG via legal suffix (end-anchored)
    for i in range(len(doc) - 1, -1, -1):
        tok = doc[i]

        if tok.text.upper() in LEGAL_SUFFIXES:
            start = i
            j = i - 1

            while j >= 0:
                t = doc[j]
                if t.is_punct or t.like_num:
                    break
                if t.ent_type_ in {"GPE", "DATE", "CARDINAL"}:
                    break
                start = j
                j -= 1

            span = doc[start:i + 1]
            if len(span) >= 2:
                return span.text

    
    # 2. Name before O/ marker
    m = O_MARKER_RE.search(doc.text)
    if m:
        before = doc.text[:m.start()].strip().split()

        name_tokens = []
        for t in reversed(before):
            if t.isalpha():
                name_tokens.append(t)
            else:
                break

        name_tokens.reverse()

        # trim trailing non-name tokens using spaCy
        while name_tokens:
            last = name_tokens[-1]
            for tok in doc:
                if tok.text == last and tok.ent_type_ in {"GPE", "LOC", "DATE", "CARDINAL"}:
                    name_tokens.pop()
                    break
            else:
                break

        if name_tokens:
            return " ".join(name_tokens)

    
    # 3. Name before STOP words or slash
    tokens = doc.text.strip().split()

    for i, tok in enumerate(tokens):
        if tok in STOP_WORDS or tok == "/":
            before = tokens[:i]

            name_tokens = []
            for t in reversed(before):
                if t.isalpha():
                    name_tokens.append(t)
                else:
                    break

            name_tokens.reverse()

            # trim trailing non-name tokens using spaCy
            while name_tokens:
                last = name_tokens[-1]
                for tok2 in doc:
                    if tok2.text == last and tok2.ent_type_ in {"GPE", "LOC", "DATE", "CARDINAL"}:
                        name_tokens.pop()
                        break
                else:
                    break

            if name_tokens:
                return " ".join(name_tokens)

    
    # 4. Fallback: spaCy (last resort)
    for ent in doc.ents:
        if ent.label_ in {"ORG", "PERSON"}:
            return ent.text

    return None


In [ ]:
texts = ["D000003886256*REALMANAGE LLC*6400 INTERNATIONAL PARKWAY STE 1000*PLANO TX 75093* 0 PLANO TX 75093 US /"
         
,"PAYCOM PAYROLL LLC 7501 W MEMORIAL RD OKLAHOMA CITY OK 73142 US /"

,"7101 CARDINAL MANAGEMENT GROUP INC 4330 PRINCE WILLIAM PKWY STE 201 WOO DBRIDGE VA 22192 /"

,"3596814 14785 PSS INDUSTRIAL GROUP CORP 10507 ELLA BLVD S TE 100 HOUSTON TX 77038 I/US"

,"A72BRWA00000000010 00012 SHERIF ABDELFATTAH O/"

,"000196455998605 ALANIZ LLC POSTAGE ACCOUNT PO BOX 799 MT PLEASANT IA 52641 US"

,"FR76300040310500010 00004 ERFI FRANCE O/2/ZA DE BILHAC O/3/FR/19120 BI LHAC"

,"ALANIZ LLC POSTAGE ACCOUNT PO BOX 799 MT PLEASANT IA 52641 US"

,"10401224 P ATTERSON UTI INTERNACIONAL COLOMBI O/BLOC OFFICE H UB QUINTO PISO STA M O/CIUDAD DEL EXTERIOR PANAMA O/PANAMA"

,"GRACE IRENE FRANCO BURBANO NOTPROVIDED /"

,"GILBERT RHODESIA GALE NOTPROVIDED /"

,"GABRIELA BALIEIRO MOREIRA RUA TIRADENTES 1720CASA2 CENTRO BR"

,"RM PROSPERITY US /"]


for text in texts:
    for delim in ['*','-']:
        text=text.replace(delim,f' {delim} ')
    doc = nlp(text)
    print(doc)

D000003886256 * REALMANAGE LLC * 6400 INTERNATIONAL PARKWAY STE 1000 * PLANO TX 75093 *  0 PLANO TX 75093 US /

REALMANAGE LLC

PAYCOM PAYROLL LLC 7501 W MEMORIAL RD OKLAHOMA CITY OK 73142 US /

PAYCOM PAYROLL LLC

7101 CARDINAL MANAGEMENT GROUP INC 4330 PRINCE WILLIAM PKWY STE 201 WOO DBRIDGE VA 22192 /

CARDINAL MANAGEMENT GROUP INC

3596814 14785 PSS INDUSTRIAL GROUP CORP 10507 ELLA BLVD S TE 100 HOUSTON TX 77038 I/US

PSS INDUSTRIAL GROUP CORP

A72BRWA00000000010 00012 SHERIF ABDELFATTAH O/

SHERIF ABDELFATTAH

000196455998605 ALANIZ LLC POSTAGE ACCOUNT PO BOX 799 MT PLEASANT IA 52641 US

ALANIZ LLC

FR76300040310500010 00004 ERFI FRANCE O/2/ZA DE BILHAC O/3/FR/19120 BI LHAC

ERFI FRANCE

ALANIZ LLC POSTAGE ACCOUNT PO BOX 799 MT PLEASANT IA 52641 US

ALANIZ LLC

10401224 P ATTERSON UTI INTERNACIONAL COLOMBI O/BLOC OFFICE H UB QUINTO PISO STA M O/CIUDAD DEL EXTERIOR PANAMA O/PANAMA

P ATTERSON UTI INTERNACIONAL COLOMBI

GRACE IRENE FRANCO BURBANO NOTPROVIDED /

GRACE IRENE FRANCO BU

In [63]:
doc = nlp("ERFI FRANCE")
for ent in doc.ents:
    print(ent,ent.label_)

ERFI ORG
FRANCE GPE
